In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import plotly.graph_objects as go
from torch.utils.data import TensorDataset, DataLoader
from typing import Callable
from tqdm.notebook import tqdm

In [31]:
torch.manual_seed(42)
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("device:", device)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("device:", device)

device: mps


In [32]:
# initial dataset and data loader
def generate_circle_data(num_samples=5000, radius=2.0):
    theta = torch.rand(num_samples) * 2 * math.pi
    x = radius * torch.cos(theta)
    y = radius * torch.sin(theta)
    return torch.stack((x, y), dim=-1)

x_clean_train = generate_circle_data(num_samples=10000)
train_ds = TensorDataset(x_clean_train)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True) 

Input: a point $(x,y)$

Output: the predicted noise $(\epsilon_x, \epsilon_y)$

From the predicted noise we're defining a direction because when we substract it we move closer to the clean data.

In [ ]:
# ARCHITECTURE
class ScoreMLP2D(nn.Module):
    def __init__(self, 
                 num_layers: int, 
                 hidden_dim: int,
                 activation: Callable[[torch.Tensor], torch.Tensor]) -> None:
        super().__init__()
        self.first_layer = nn.Linear(in_features=2, out_features=hidden_dim)
        self.layers = nn.ModuleList() 
        for i in range(num_layers):
            self.layers.append(
                nn.Linear(in_features=hidden_dim, out_features=hidden_dim)
            )
        self.activation = activation
        self.last_layer = nn.Linear(in_features=hidden_dim, out_features=2)

    def forward(self, meshgrid: torch.Tensor) -> torch.Tensor:
        out = meshgrid
        out = self.first_layer(out)
        for layer in self.layers:
            out = layer(out)
            out = self.activation(out)
        out = self.last_layer(out)
        return out

model = ScoreMLP2D(num_layers=3, hidden_dim=128, activation=torch.nn.functional.elu)
model = model.to(device)
opt = optim.Adam(model.parameters(), lr=0.001)

In [37]:
#training parameters
sigma = 0.3 
epochs = 6#50

In [ ]:
for epoch in tqdm(range(epochs), desc="epoch"):
    model.train()
    epoch_loss = 0.0  
    num_batches = 0   
    for (xb,) in train_dl:
        x_clean = xb.to(device)
        noise = torch.randn_like(x_clean)
        # here I dirty the data with noise
        x_noisy = x_clean + sigma * noise
        pred_noise = model(x_noisy)
        #I compute the mse loss
        loss = F.mse_loss(pred_noise, noise)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_loss += loss.item()
        num_batches += 1
        
    # mean loss in epoch
    print(epoch, epoch_loss / num_batches)
    #if (epoch + 1) % 10 == 0 or epoch == 0:
    #    print(epoch, epoch_loss / num_batches)

epoch:   0%|          | 0/6 [00:00<?, ?it/s]

0 0.9763558704382295
1 0.7556443833241797
2 0.5460483023695125
3 0.5356060236122957
4 0.5406556816617395
5 0.5377848712122364


Given the equations in the pdf I can get the score as a function of the noise.

- $\nabla_{\tilde{\mathbf{x}}} \log q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x}) = \frac{1}{\sigma^2}(\mathbf{x} - \tilde{\mathbf{x}})$

- $\tilde{\mathbf{x}} = \mathbf{x} + \sigma \epsilon \implies \mathbf{x} - \tilde{\mathbf{x}} = -\sigma \epsilon$

Here:
* **$\mathbf{x}$** is the original data
* **$\tilde{\mathbf{x}}$** represents noisy data
* **$\sigma$** is the SD of the Gaussian noise
* **$\epsilon$** is the standard Gaussian noise vector that will dirty my data
* **$q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x})$** is the conditional probability distribution of the noisy data given the clean data and will model the noise corruption process.
* **$\nabla_{\tilde{\mathbf{x}}}$**: the gradient is computed wrt the noisy data and provides the direction of change

In the end I get
$$\nabla_{\tilde{\mathbf{x}}} \log q_\sigma(\tilde{\mathbf{x}} \mid \mathbf{x}) = \frac{-\sigma \epsilon}{\sigma^2} = -\frac{\epsilon}{\sigma}$$

Then I implement Langevin dynamics `x_langevin` from the following equation
$$\tilde{\mathbf{x}}^k = \tilde{\mathbf{x}}^{k-1} + \frac{\lambda_i}{2} \mathbf{s}_\theta(\tilde{\mathbf{x}}^{k-1}, \sigma_i) + \sqrt{\lambda_i}\mathbf{z}^k$$

where at each step $k$ I update the previous position $\tilde{\mathbf{x}}^{k-1}$ moving along the direction of the score with a step length scaled by $\lambda_i$. A standard Gaussian vector $\mathbf{z}^k$ adds randomness to explore the space.

In [ ]:
# langevin sampling
model.eval()
num_steps = 300
step_size = 0.01  

x_langevin = torch.randn(1000, 2, device=device) * 4.0 
initial_noise_np = x_langevin.cpu().clone().numpy()

with torch.no_grad():
    for k in range(num_steps):
        pred_noise = model(x_langevin)
        score = - pred_noise / sigma
        #langevin dynamics step
        z = torch.randn_like(x_langevin)
        x_langevin = x_langevin + 0.5 * step_size * score + math.sqrt(step_size) * z

final_generated_np = x_langevin.cpu().numpy()

In [41]:
#visualization
x_clean_np = x_clean_train.cpu().numpy()
def plot_2d_points(points, title, color):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=points[:, 0], y=points[:, 1],
            mode='markers',
            marker=dict(size=4, color=color, opacity=0.6),
            name=title
        )
    )
    fig.update_layout(
        title=title, width=600, height=600,
        xaxis=dict(range=[-6, 6]), yaxis=dict(range=[-6, 6])
    )
    fig.show()

plot_2d_points(x_clean_np, "Original data", "lightgreen")
plot_2d_points(initial_noise_np, "Noise", "red")
plot_2d_points(final_generated_np, "Data generated", "orange")